This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path
from enum import Enum
import csv
from decimal import Decimal

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences, savgol_filter

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, weight_factor, stopping_criteria, _nan_divide

### Functions

In [ ]:
def get_graphable_data(phi: NDHistogram) -> tuple[np.ndarray, np.ndarray]:
    """Get data from phi in graphable format.

    Counts and energy bin midpoints will be extracted as 1D arrays.
    :param phi: Neutron energy spectrum
    :type phi: NDHistogram
    :return: Counts array, energy bin midpoints array
    :rtype: tuple[np.ndarray, np.ndarray]
    """
    return phi.counts.reshape(-1), phi.midpoints[-1]

In [ ]:
def get_next_multiple(value: float, base: float) -> float:
    """Get smallest multiple of base that is greater than value"""
    if value < 0:
        raise ValueError("Value must not be negative")
    if base <= 0:
        raise ValueError("Base must be positive")

    value_decimal = Decimal(str(value))
    base_decimal = Decimal(str(base))
    remainder = value_decimal % base_decimal
    return float(value_decimal + (base_decimal - remainder))

## Setting Entry

In [ ]:
energies = np.arange(0.124, 6.014, step=0.062)
exp_data = {f"{energy:.3f}_MeV": {} for energy in energies}

In [ ]:
bins_min = 0
bins_max = 6
bins_width = 0.02

In [ ]:
max_iters = 50

## Data Processing

### Loading

In [ ]:
R = load_neutron_response_matrix(
    # Path("response_matrix_R4_mono"),
    Path("response_matrix_11MeV"),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
bins = np.arange(bins_min, bins_max + bins_width, bins_width)

base_path = Path("response_matrix_R4_mono")
for energy, eng_data in exp_data.items():
    filepath = base_path / f"neutron_{energy}.csv.npy"
    eng_data["filepath"] = filepath

    L_array = np.load(filepath)

    np_cps, *_ = np.histogram(L_array, bins=bins)

    mov_avg_window = 7
    polyorder = 3
    np_cps_savgol = savgol_filter(np_cps, window_length=mov_avg_window, polyorder=polyorder)

    np_cps = np_cps.reshape(-1, 1)
    np_cps_savgol = np_cps_savgol.reshape(-1, 1)
    np_Ls = (bins[1:] + bins[:-1]) / 2

    if energy == "1.302_MeV":
        pass

    N = NDHistogram(np_cps_savgol, [np_Ls, np.ones(1)])
    eng_data["N"] = N

### Processing

In [ ]:
for energy, eng_data in exp_data.items():
    print(energy)
    N = eng_data["N"]
    phi, unfold_info = unfold_spectrum(
        R,
        N,
        L_cut=0.05,
        max_iterations=max_iters,
        full_info=True
    )
    eng_data["phi"] = phi
    eng_data["unfold_info"] = unfold_info

### Uncertainty Estimation

#### Horizontal

In [ ]:
sigma = 0.050   # MeVee
shift = get_next_multiple(3*sigma, bins_width)
print(3*sigma, shift)

In [ ]:
lshift_R = load_neutron_response_matrix(
    Path("response_matrix_R4_mono"),
    min_L=0,
    max_L=bins_max-shift,
    L_bin_widths=bins_width
)
for energy, eng_data in exp_data.items():
    N = eng_data["N"]
    lshift_mids = N.midpoints[0] - shift
    nonzero_mask = lshift_mids > 0
    lshift_N = NDHistogram(N.counts[nonzero_mask], [lshift_mids[nonzero_mask], N.midpoints[1]])

    n_mids = lshift_N.midpoints[0]
    r_mids = lshift_R.midpoints[0]
    if n_mids.shape == r_mids.shape and np.isclose(n_mids, r_mids).all():
        lshift_N = NDHistogram(lshift_N.counts, [r_mids, lshift_N.midpoints[1]])
    else:
        raise ValueError(f"R and N midpoints don't match on energy {energy}")

    lshift_phi, *_ = unfold_spectrum(
        lshift_R,
        lshift_N,
        max_iterations=max_iters
    )
    eng_data["lshift_N"] = lshift_N
    eng_data["lshift_phi"] = lshift_phi

In [ ]:
rshift_R = load_neutron_response_matrix(
    Path("response_matrix_R4_mono"),
    min_L=bins_min+shift,
    max_L=bins_max+shift,
    L_bin_widths=bins_width
)

for energy, eng_data in exp_data.items():
    N = eng_data["N"]
    rshift_mids = N.midpoints[0] + shift
    rshift_N = NDHistogram(N.counts, [rshift_mids, N.midpoints[1]])

    if np.isclose(rshift_N.midpoints[0], rshift_R.midpoints[0]).all():
        rshift_N = NDHistogram(rshift_N.counts, [rshift_R.midpoints[0], rshift_N.midpoints[1]])
    else:
        raise ValueError("R and N midpoints don't match")

    rshift_phi, *_ = unfold_spectrum(
        rshift_R,
        rshift_N,
        max_iterations=50
    )
    eng_data["rshift_N"] = rshift_N
    eng_data["rshift_phi"] = rshift_phi

### Lowpass Filtering

In [ ]:
from scipy.signal import butter, lfilter, filtfilt


def butter_lowpass(cutoff, fs, order=5):
    return butter(order, cutoff, fs=fs, btype='low', analog=False)


def butter_lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    # y = lfilter(b, a, data)
    y = filtfilt(b, a, data)
    return y


order = 6
period = 0.062
fs = 1/period
cutoff = fs/4

for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts)
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts)
    lmids = lshift_phi.midpoints[1]
    wrong_side_mask = lmids <= energy_num
    wrong_side_mask = wrong_side_mask.reshape((1, -1))
    rshift_phi_counts[wrong_side_mask] = 0
    
    lshift_counts_filtered = butter_lowpass_filter(lshift_phi_counts, cutoff, fs, order)
    rshift_counts_filtered = butter_lowpass_filter(rshift_phi_counts, cutoff, fs, order)
    eng_data["lshift_phi_filtered"] = NDHistogram(lshift_counts_filtered, lshift_phi.midpoints)
    eng_data["rshift_phi_filtered"] = NDHistogram(rshift_counts_filtered, rshift_phi.midpoints)

### Peak Finding

In [ ]:
energy_limits = (1.5, 5.5)

### Plotting (Diagnostic)

In [ ]:
figsize = (15,6)
# N (with L and R shift versions)
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    if energy_num > 5.5 or energy_num < 1.5:
        continue
    print(energy)
    N = eng_data["N"]
    lshift_N = eng_data["lshift_N"]
    rshift_N = eng_data["rshift_N"]

    phi = eng_data["phi"]
    lshift_phi = eng_data["lshift_phi"]
    rshift_phi = eng_data["rshift_phi"]
    lshift_phi_counts = np.nan_to_num(lshift_phi.counts.reshape(-1))
    rshift_phi_counts = np.nan_to_num(rshift_phi.counts.reshape(-1))
    lmids = lshift_phi.midpoints[1]
    energy_num = float(energy[:5])
    wrong_side_mask = lmids <= energy_num
    rshift_phi_counts[wrong_side_mask] = 0

    lshift_phi_filtered = eng_data["lshift_phi_filtered"]
    rshift_phi_filtered = eng_data["rshift_phi_filtered"]

    fig, axs = plt.subplots(1, 3, figsize=figsize)
    ax1, ax2, ax3 = axs
    
    ax1.plot(N.midpoints[0], N.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax1.plot(lshift_N.midpoints[0], lshift_N.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax1.plot(rshift_N.midpoints[0], rshift_N.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax1.set(xlabel="L (MeVee)", ylabel="Counts", title="PHD", yscale="symlog", ylim=(1, 1e4))
    ax1.legend()

    ax2.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax2.plot(lshift_phi.midpoints[1], lshift_phi_counts, linestyle="dashed", label="Left shift")
    ax2.plot(rshift_phi.midpoints[1], rshift_phi_counts, linestyle="dashed", label="Right shift")
    ax2.set(
        xlabel="E (MeV)", ylabel="Counts",
        title="Phi (No Filter)"
    )
    ax2.legend()

    ax3.plot(phi.midpoints[1], phi.counts.reshape(-1), linestyle="solid", label="Unshifted")
    ax3.plot(lshift_phi_filtered.midpoints[1], lshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Left shift")
    ax3.plot(rshift_phi_filtered.midpoints[1], rshift_phi_filtered.counts.reshape(-1), linestyle="dashed", label="Right shift")
    ax3.set(
        xlabel="E (MeV)", ylabel="Counts",
        title="Phi (Filtered)"
    )
    ax3.legend()

    fig.suptitle(energy)

    plt.show()

### Peak Finding

In [ ]:
prominence = 0.01

In [ ]:
for energy, eng_data in exp_data.items():
    phi = eng_data["phi"]
    phi_flat, phi_mids = get_graphable_data(phi)
    if np.isnan(phi_flat).all():
        eng_data["peak_e"] = None
        continue
    peaks, p_data = find_peaks(
        phi_flat,
        prominence=prominence
    )
    if len(peaks) > 0:
        idx_peak = phi_flat[peaks].argmax()
        peak_e = phi_mids[peaks][idx_peak]
    else:
        peak_e = None
    eng_data["peak_e"] = peak_e

In [ ]:
for energy, eng_data in exp_data.items():
    peak_key = "lshift_peak_e"
    lshift_phi = eng_data["lshift_phi"]
    lshift_phi_flat, lshift_phi_mids = get_graphable_data(lshift_phi)
    if np.isnan(lshift_phi_flat).all():
        eng_data[peak_key] = None
        continue
        
    lpeaks, lp_data = find_peaks(
        lshift_phi_flat,
        prominence=prominence
    )
    if len(lpeaks) <= 0:
        eng_data[peak_key] = None
        continue
    
    peak_e = eng_data["peak_e"]
    if peak_e is None:
        eng_data[peak_key] = None
        continue

    peak_energies = lshift_phi_mids[lpeaks]
    peak_heights = lshift_phi_flat[lpeaks]
    lo_energy_mask = peak_energies < peak_e
    lo_energy_peak_heights = peak_heights[lo_energy_mask]
    lo_peak_energies = peak_energies[lo_energy_mask]
    if len(lo_energy_peak_heights) <= 0 or len(lo_peak_energies) <= 0:
        eng_data[peak_key] = None
        continue

    idx_peak = lo_energy_peak_heights.argmax()
    lshift_peak_e = lo_peak_energies[idx_peak]
    eng_data["lshift_peak_e"] = lshift_peak_e

In [ ]:
for energy, eng_data in exp_data.items():
    peak_key = "rshift_peak_e"
    rshift_phi = eng_data["rshift_phi"]
    rshift_phi_flat, rshift_phi_mids = get_graphable_data(rshift_phi)
    if np.isnan(rshift_phi_flat).all():
        eng_data[peak_key] = None
        continue
        
    rpeaks, rp_data = find_peaks(
        rshift_phi_flat,
        prominence=prominence
    )
    if len(rpeaks) <= 0:
        eng_data[peak_key] = None
        continue
    
    peak_e = eng_data["peak_e"]
    if peak_e is None:
        eng_data[peak_key] = None
        continue

    peak_energies = rshift_phi_mids[rpeaks]
    peak_heights = rshift_phi_flat[rpeaks]
    hi_energy_mask = peak_energies > peak_e
    hi_energy_peak_heights = peak_heights[hi_energy_mask]
    hi_peak_energies = peak_energies[hi_energy_mask]
    if len(hi_energy_peak_heights) <= 0 or len(hi_peak_energies) <= 0:
        eng_data[peak_key] = None
        continue
    
    idx_peak = hi_energy_peak_heights.argmax()
    rshift_peak_e = hi_peak_energies[idx_peak]
    eng_data["rshift_peak_e"] = rshift_peak_e

In [ ]:
uncertainties = []
for energy, eng_data in exp_data.items():
    energy_num = float(energy[:5])
    peak_e = eng_data["peak_e"]
    lshift_peak_e = eng_data["lshift_peak_e"]
    rshift_peak_e = eng_data["rshift_peak_e"]
    
    has_peak_e = peak_e is not None
    has_lo_uncert = has_peak_e and lshift_peak_e is not None
    has_hi_uncert = has_peak_e and rshift_peak_e is not None
    uncert_lo = float(peak_e - lshift_peak_e) if has_lo_uncert else None
    uncert_hi = float(rshift_peak_e - peak_e) if has_hi_uncert else None
    
    e_uncert = uncert_lo, uncert_hi
    uncertainties.append((
        energy_num,
        uncert_lo,
        uncert_hi
    ))
    eng_data["e_uncertianty"] = e_uncert
uncertainties = [
    uncertainty for uncertainty in uncertainties
    if uncertainty[-2] is not None and uncertainty[-1] is not None
]
uncertainties

In [ ]:
with open(f"unfolding_uncertainties_{bins_min}-{bins_max}-{bins_width}.csv", 'w') as csvfile:
    fieldnames = ["energy", "left_unc", "right_unc"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for energy, left_unc, right_unc in uncertainties:
        row = {"energy": energy, "left_unc": left_unc, "right_unc": right_unc}
        writer.writerow(row)

## Plotting

In [ ]:
figsize = (9, 6)
fontsize = 20

In [ ]:
plot_data = [(float(energy[:4]), eng_data["e_uncertianty"][0], eng_data["e_uncertianty"][1]) for energy, eng_data in exp_data.items()]
energies, lo_uncerts, hi_uncerts = zip(*plot_data)
energies = list(energies)[2:]
lo_uncerts = list(lo_uncerts)[2:]
hi_uncerts = list(hi_uncerts)[2:]

fig, ax = plt.subplots(figsize=figsize)
ax.plot(energies, lo_uncerts, marker="o", markersize=3, label="Lo")
ax.plot(energies, hi_uncerts, marker="o", markersize=3, label="Hi")
ax.set(xlabel="Energy (MeV)", ylabel="Uncertainty (MeV)")
ax.legend()
plt.show()

In [ ]:
energy = "4.526_MeV"
eng_data = exp_data[energy]
phi = eng_data["phi"]
lshift_phi = eng_data["lshift_phi"]
rshift_phi = eng_data["rshift_phi"]
phi_flat, phi_mids = get_graphable_data(phi)
l_phi_flat, l_phi_mids = get_graphable_data(lshift_phi)
r_phi_flat, r_phi_mids = get_graphable_data(rshift_phi)

fig, ax = plt.subplots(figsize=figsize)
ax.plot(phi_mids, phi_flat, marker="o", markersize=3, label="Normal")
ax.plot(l_phi_mids, l_phi_flat, marker="o", markersize=3, label="Lo")
ax.plot(r_phi_mids, r_phi_flat, marker="o", markersize=3, label="Hi")
ax.set(xlabel="Energy (MeV)", ylabel="Uncertainty (MeV)")
ax.set_ylim(0, 0.15)
ax.legend()
plt.show()

## End

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()